# 3. Push Fine-Tuned Model to HuggingFace Hub

This notebook takes the fine-tuned sentence transformer from notebook 02 and:
1. Generates a model card (README.md) with training details and metrics
2. Pushes the model to HuggingFace Hub
3. Optionally exports to ONNX for faster inference

## What you need
- A **trained model** directory (from notebook 02)
- A **HuggingFace account** (free at [huggingface.co](https://huggingface.co))

## 1. Install Dependencies

In [ ]:
!pip install -q sentence-transformers huggingface_hub

## 2. Setup

In [ ]:
import json
import os
from pathlib import Path

import torch
from sentence_transformers import SentenceTransformer

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")

## 3. Configuration

In [ ]:
# === HuggingFace ===
HF_USERNAME = "your-username"                        # your HuggingFace username
MODEL_NAME = "my-sentence-transformer"               # model name on the Hub
PRIVATE = False                                      # set True for private model

# === Paths ===
MODEL_DIR = "/content/drive/MyDrive/sentence-transformer-finetune/trained_model"

# Derived
HF_REPO = f"{HF_USERNAME}/{MODEL_NAME}"

print(f"Model dir:  {MODEL_DIR}")
print(f"HF repo:    {HF_REPO}")
print(f"Private:    {PRIVATE}")

## 4. Load Training Metadata & Metrics

In [ ]:
# Load metadata
metadata_path = os.path.join(MODEL_DIR, "training_metadata.json")
if os.path.exists(metadata_path):
    with open(metadata_path, "r") as f:
        metadata = json.load(f)
    print("Training metadata:")
    print(json.dumps(metadata, indent=2))
else:
    metadata = {}
    print("No training_metadata.json found — model card will have limited details")

# Load comparison metrics
comparison_path = os.path.join(MODEL_DIR, "comparison.json")
if os.path.exists(comparison_path):
    with open(comparison_path, "r") as f:
        comparison = json.load(f)
    print("\nComparison metrics loaded")
else:
    comparison = {}
    print("\nNo comparison.json found")

# Load eval results
eval_path = os.path.join(MODEL_DIR, "eval_results.json")
if os.path.exists(eval_path):
    with open(eval_path, "r") as f:
        eval_results = json.load(f)
    print("Eval results loaded")
else:
    eval_results = {}

## 5. Generate Model Card

In [ ]:
def build_metrics_table(eval_results: dict) -> str:
    """Build a markdown table from evaluation results."""
    if not eval_results:
        return "_No evaluation results available._"

    lines = ["| Metric | Score |", "|--------|-------|"]
    for key, val in sorted(eval_results.items()):
        if isinstance(val, (int, float)):
            lines.append(f"| {key} | {float(val):.4f} |")
    return "\n".join(lines)


def build_comparison_table(comparison: dict) -> str:
    """Build a markdown comparison table."""
    if not comparison:
        return ""

    lines = [
        "\n### Base vs Fine-Tuned\n",
        "| Metric | Base | Fine-Tuned | Improvement |",
        "|--------|------|------------|-------------|",
    ]
    for metric, vals in comparison.items():
        base = vals.get("base", 0)
        ft = vals.get("finetuned", 0)
        diff = ft - base
        pct = (diff / base * 100) if base > 0 else 0
        lines.append(f"| {metric} | {base:.4f} | {ft:.4f} | +{diff:.4f} ({pct:+.1f}%) |")
    return "\n".join(lines)


base_model = metadata.get("base_model", "sentence-transformers/all-MiniLM-L6-v2")
max_seq = metadata.get("max_seq_length", 512)
train_samples = metadata.get("train_samples", "N/A")
epochs = metadata.get("epochs", "N/A")
batch_size = metadata.get("batch_size", "N/A")
lr = metadata.get("learning_rate", "N/A")

model_card = f"""---
tags:
- sentence-transformers
- feature-extraction
- sentence-similarity
language:
- en
license: apache-2.0
library_name: sentence-transformers
pipeline_tag: sentence-similarity
---

# {MODEL_NAME}

Fine-tuned version of [{base_model}](https://huggingface.co/{base_model}) for semantic similarity and retrieval tasks.

## Training Details

| Parameter | Value |
|-----------|-------|
| Base model | `{base_model}` |
| Max sequence length | {max_seq} |
| Training samples | {train_samples} |
| Epochs | {epochs} |
| Batch size | {batch_size} |
| Learning rate | {lr} |
| Loss | MultipleNegativesRankingLoss |

## Evaluation Results

{build_metrics_table(eval_results)}

{build_comparison_table(comparison)}

## Usage

```python
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("{HF_REPO}")

# Encode texts
embeddings = model.encode(["your text here", "another text"])

# Compute similarity
from sentence_transformers import util
similarity = util.cos_sim(embeddings[0], embeddings[1])
```

## License

Apache 2.0
"""

# Save model card
model_card_path = os.path.join(MODEL_DIR, "README.md")
with open(model_card_path, "w") as f:
    f.write(model_card)

print("Model card generated:")
print(model_card[:500] + "...")

## 6. Authenticate & Push to HuggingFace

In [ ]:
from huggingface_hub import notebook_login, whoami

print("Login to HuggingFace:")
notebook_login()

In [ ]:
# Verify authentication
try:
    user_info = whoami()
    print(f"Authenticated as: {user_info['name']}")
except Exception:
    raise RuntimeError("Not authenticated — run the login cell above")

if HF_USERNAME == "your-username":
    raise ValueError("Set HF_USERNAME in the configuration cell above")

In [ ]:
print(f"\nPushing model to: {HF_REPO}")
print(f"Model directory: {MODEL_DIR}")
print(f"Private: {PRIVATE}\n")

model = SentenceTransformer(MODEL_DIR)

model.push_to_hub(
    HF_REPO,
    private=PRIVATE,
    commit_message=f"Upload fine-tuned model ({train_samples} samples, {epochs} epochs)",
    exist_ok=True,
)

print(f"\nModel published: https://huggingface.co/{HF_REPO}")

del model

## 7. (Optional) Export to ONNX

ONNX export enables faster CPU inference and compatibility with tools like Transformers.js.

In [ ]:
!pip install -q optimum[onnxruntime] onnx

In [ ]:
try:
    from optimum.onnxruntime import ORTModelForFeatureExtraction
    from transformers import AutoTokenizer
    from huggingface_hub import HfApi
    import numpy as np

    onnx_dir = os.path.join(MODEL_DIR, "onnx")
    Path(onnx_dir).mkdir(parents=True, exist_ok=True)

    print(f"Exporting {HF_REPO} to ONNX...")
    ort_model = ORTModelForFeatureExtraction.from_pretrained(
        HF_REPO,
        export=True,
        provider="CPUExecutionProvider",
    )

    ort_model.save_pretrained(onnx_dir)
    tokenizer = AutoTokenizer.from_pretrained(HF_REPO)
    tokenizer.save_pretrained(onnx_dir)

    # Test
    inputs = tokenizer(["test sentence"], return_tensors="pt")
    outputs = ort_model(**inputs)
    print(f"ONNX export successful — output shape: {outputs.last_hidden_state.shape}")

    # Get size
    onnx_files = list(Path(onnx_dir).glob("*.onnx"))
    if onnx_files:
        size_mb = sum(f.stat().st_size for f in onnx_files) / (1024 * 1024)
        print(f"ONNX model size: {size_mb:.1f} MB")

    # Upload to Hub
    print(f"\nUploading ONNX to {HF_REPO}...")
    api = HfApi()
    api.upload_folder(
        folder_path=onnx_dir,
        path_in_repo="onnx",
        repo_id=HF_REPO,
        repo_type="model",
        commit_message="Add ONNX export",
    )
    print("ONNX uploaded successfully")

    del ort_model

except ImportError:
    print("Install optimum for ONNX export: pip install optimum[onnxruntime]")
    print("Skipping — the PyTorch model is still available on the Hub.")
except Exception as e:
    print(f"ONNX export failed: {e}")
    print("The PyTorch model is still available on the Hub.")

## Done!

Your model is published at `https://huggingface.co/{HF_REPO}`.

Load it anywhere with:
```python
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("{HF_REPO}")
```